# YouTube Title Extractor

Extract titles from top-performing YouTube videos — no coding required.

## How to use this tool

1. **Run Step 1** (click the play button) — this installs the scraper. Only needed once.
2. **Pick your mode** in Step 2 — fill in the form, then click play.
3. **See your results** below the cell. Optionally download as CSV.

---

## Step 1: Setup (run this once)
Click the **play button** on the left side of the cell below. Wait until it says **"Ready!"**

In [ ]:
# ============================
# STEP 1: SETUP (run once)
# ============================
!pip install -q yt-dlp

import subprocess
import json
import csv
import os
from datetime import datetime
from urllib.parse import parse_qs, urlparse
from IPython.display import display, HTML, clear_output
from google.colab import files as colab_files


def format_number(n):
    if n is None:
        return "N/A"
    if n >= 1_000_000_000:
        return f"{n / 1_000_000_000:.1f}B"
    if n >= 1_000_000:
        return f"{n / 1_000_000:.1f}M"
    if n >= 1_000:
        return f"{n / 1_000:.1f}K"
    return str(n)


def format_duration(seconds):
    if seconds is None:
        return "N/A"
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    if hours > 0:
        return f"{hours}:{minutes:02d}:{secs:02d}"
    return f"{minutes}:{secs:02d}"


def format_date(date_str):
    if not date_str or len(date_str) != 8:
        return "N/A"
    return f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}"


def run_yt_dlp(args):
    cmd = ["yt-dlp", "--dump-json", "--no-download", "--no-warnings"] + args
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        if result.returncode == 0 and result.stdout.strip():
            entries = []
            for line in result.stdout.strip().split("\n"):
                line = line.strip()
                if line:
                    try:
                        entries.append(json.loads(line))
                    except json.JSONDecodeError:
                        continue
            return entries
    except subprocess.TimeoutExpired:
        print("  Request timed out, try again.")
    except Exception as e:
        print(f"  Error: {e}")
    return []


def extract_video_info(entry):
    return {
        "title": entry.get("title", "N/A"),
        "url": entry.get("webpage_url") or entry.get("url") or entry.get("original_url", "N/A"),
        "video_id": entry.get("id", "N/A"),
        "channel": entry.get("channel") or entry.get("uploader", "N/A"),
        "channel_subscribers": entry.get("channel_follower_count"),
        "views": entry.get("view_count"),
        "likes": entry.get("like_count"),
        "comments": entry.get("comment_count"),
        "duration": entry.get("duration"),
        "upload_date": entry.get("upload_date"),
    }


def sort_videos(videos, sort_by):
    if sort_by == "Most Views":
        return sorted(videos, key=lambda v: v.get("views") or 0, reverse=True)
    elif sort_by == "Newest First":
        return sorted(videos, key=lambda v: v.get("upload_date") or "0", reverse=True)
    elif sort_by == "Most Engagement (likes+comments)":
        def engagement(v):
            likes = v.get("likes") or 0
            comments = v.get("comments") or 0
            views = v.get("views") or 1
            return (likes + comments * 3) / views
        return sorted(videos, key=engagement, reverse=True)
    elif sort_by == "Most Subscribers":
        return sorted(videos, key=lambda v: v.get("channel_subscribers") or 0, reverse=True)
    return videos


def show_results_html(videos, titles_only=False):
    if not videos:
        display(HTML("<h3 style='color:red;'>No videos found. Try a different input.</h3>"))
        return

    if titles_only:
        html = f"<h3>Titles ({len(videos)} videos)</h3><ol>"
        for v in videos:
            html += f"<li style='margin-bottom:4px; font-size:14px;'><b>{v['title']}</b></li>"
        html += "</ol>"
        display(HTML(html))
        return

    html = f"<h3>Results ({len(videos)} videos)</h3>"
    html += "<table style='border-collapse:collapse; width:100%; font-size:13px;'>"
    html += "<tr style='background:#f0f0f0; text-align:left;'>"
    html += "<th style='padding:8px; border:1px solid #ddd;'>#</th>"
    html += "<th style='padding:8px; border:1px solid #ddd;'>Title</th>"
    html += "<th style='padding:8px; border:1px solid #ddd;'>Channel</th>"
    html += "<th style='padding:8px; border:1px solid #ddd;'>Views</th>"
    html += "<th style='padding:8px; border:1px solid #ddd;'>Likes</th>"
    html += "<th style='padding:8px; border:1px solid #ddd;'>Duration</th>"
    html += "<th style='padding:8px; border:1px solid #ddd;'>Uploaded</th>"
    html += "<th style='padding:8px; border:1px solid #ddd;'>Subs</th>"
    html += "</tr>"

    for i, v in enumerate(videos, 1):
        bg = '#ffffff' if i % 2 == 1 else '#f9f9f9'
        url = v.get('url', '#')
        html += f"<tr style='background:{bg};'>"
        html += f"<td style='padding:6px; border:1px solid #ddd; text-align:center;'>{i}</td>"
        html += f"<td style='padding:6px; border:1px solid #ddd;'><a href='{url}' target='_blank'>{v['title']}</a></td>"
        html += f"<td style='padding:6px; border:1px solid #ddd;'>{v['channel']}</td>"
        html += f"<td style='padding:6px; border:1px solid #ddd; text-align:right;'>{format_number(v['views'])}</td>"
        html += f"<td style='padding:6px; border:1px solid #ddd; text-align:right;'>{format_number(v['likes'])}</td>"
        html += f"<td style='padding:6px; border:1px solid #ddd; text-align:center;'>{format_duration(v['duration'])}</td>"
        html += f"<td style='padding:6px; border:1px solid #ddd; text-align:center;'>{format_date(v['upload_date'])}</td>"
        html += f"<td style='padding:6px; border:1px solid #ddd; text-align:right;'>{format_number(v['channel_subscribers'])}</td>"
        html += "</tr>"

    html += "</table>"
    display(HTML(html))


def export_csv_and_download(videos, filename="youtube_titles.csv"):
    if not videos:
        print("No data to export.")
        return
    fieldnames = ["rank", "title", "channel", "subscribers", "views", "likes",
                  "comments", "duration_seconds", "upload_date", "url"]
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for i, v in enumerate(videos, 1):
            writer.writerow({
                "rank": i, "title": v["title"], "channel": v["channel"],
                "subscribers": v["channel_subscribers"] or "",
                "views": v["views"] or "", "likes": v["likes"] or "",
                "comments": v["comments"] or "",
                "duration_seconds": v["duration"] or "",
                "upload_date": format_date(v["upload_date"]), "url": v["url"],
            })
    colab_files.download(filename)
    print(f"Downloading {filename}...")


print("\n" + "=" * 50)
print("  Setup complete! Ready to use.")
print("=" * 50)
print("\nNow go to Step 2 below.")

---
## Step 2: Extract YouTube Titles

1. **Fill in the form** on the right side (click the form fields)
2. **Click the play button** to run

**Choose your mode:**
- `Video URL` — paste a direct YouTube video link
- `Search Results URL` — paste a YouTube search page URL
- `Keyword Search` — just type a topic/keyword


In [ ]:
# =============================================
# STEP 2: FILL IN THE FORM AND CLICK PLAY
# =============================================

#@title  { run: "auto", display-mode: "form" }

#@markdown ### Choose your mode and enter your input:

mode = "Keyword Search"  #@param ["Video URL", "Search Results URL", "Keyword Search"]
your_input = ""  #@param {type:"string"}

#@markdown ---
#@markdown ### Options:
sort_by = "Relevance (YouTube default)"  #@param ["Relevance (YouTube default)", "Most Views", "Newest First", "Most Engagement (likes+comments)", "Most Subscribers"]
number_of_results = 30  #@param {type:"slider", min:5, max:50, step:5}
show_titles_only = False  #@param {type:"boolean"}

# ---- Processing ----
if not your_input.strip():
    display(HTML("<h3 style='color:orange;'>Please enter a URL or keyword in the form above.</h3>"))
else:
    input_text = your_input.strip()
    videos = []

    if mode == "Video URL":
        print(f"Fetching title for: {input_text}")
        # Support multiple URLs separated by commas or newlines
        urls = [u.strip() for u in input_text.replace(",", "\n").split("\n") if u.strip()]
        for url in urls:
            print(f"  Fetching: {url}")
            entries = run_yt_dlp([url])
            for entry in entries:
                videos.append(extract_video_info(entry))

    elif mode == "Search Results URL":
        parsed = urlparse(input_text)
        params = parse_qs(parsed.query)
        query = params.get("search_query", [None])[0]
        if not query:
            if "/hashtag/" in parsed.path:
                query = parsed.path.split("/hashtag/")[-1]
        if not query:
            display(HTML("<h3 style='color:red;'>Could not extract search query from that URL.</h3>"))
            display(HTML("<p>Expected format: <code>https://www.youtube.com/results?search_query=your+search</code></p>"))
        else:
            print(f"Searching YouTube for: \"{query}\"")
            print(f"Fetching top {number_of_results} results...")
            search_entries = run_yt_dlp([f"ytsearch{number_of_results}:{query}", "--flat-playlist"])
            if search_entries:
                print(f"Found {len(search_entries)} results. Getting details...")
                for entry in search_entries:
                    vid_id = entry.get("id") or entry.get("url")
                    if vid_id:
                        url = vid_id if vid_id.startswith("http") else f"https://www.youtube.com/watch?v={vid_id}"
                        details = run_yt_dlp([url])
                        for d in details:
                            videos.append(extract_video_info(d))
                            print(f"  [{len(videos)}/{len(search_entries)}] {extract_video_info(d)['title'][:60]}")

    elif mode == "Keyword Search":
        print(f"Searching YouTube for: \"{input_text}\"")
        print(f"Fetching top {number_of_results} results...")
        search_entries = run_yt_dlp([f"ytsearch{number_of_results}:{input_text}", "--flat-playlist"])
        if search_entries:
            print(f"Found {len(search_entries)} results. Getting details...")
            for entry in search_entries:
                vid_id = entry.get("id") or entry.get("url")
                if vid_id:
                    url = vid_id if vid_id.startswith("http") else f"https://www.youtube.com/watch?v={vid_id}"
                    details = run_yt_dlp([url])
                    for d in details:
                        videos.append(extract_video_info(d))
                        print(f"  [{len(videos)}/{len(search_entries)}] {extract_video_info(d)['title'][:60]}")

    # Sort
    videos = sort_videos(videos, sort_by)

    # Display
    print("\n")
    show_results_html(videos, titles_only=show_titles_only)

    # Store for CSV export
    _last_results = videos

---
## Step 3 (Optional): Download results as a spreadsheet

Run the cell below to download your results as a `.csv` file.  
You can open `.csv` files in **Excel** or **Google Sheets**.

In [ ]:
# =============================================
# STEP 3: DOWNLOAD AS CSV (optional)
# =============================================

#@title Click play to download results as CSV { display-mode: "form" }
filename = "youtube_titles.csv"  #@param {type:"string"}

try:
    if _last_results:
        export_csv_and_download(_last_results, filename)
    else:
        print("No results to export. Run Step 2 first.")
except NameError:
    print("No results to export. Run Step 2 first.")